# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print("Dataset Name: ", metadata.name)
print("Description: ")
print(metadata.description)


## 2. Data Overview
Review available record sets, fields, and their `@id` identifiers.

In [ ]:
# Display available record sets
print("Available record sets (by @id):")
for record_set in metadata.record_sets:
    print(f"- @id: {record_set.id}, name: {record_set.name}")

# For demonstration, print fields for each record set
for record_set in metadata.record_sets:
    print(f"\nRecord set '@id': {record_set.id}")
    print("Fields (@id and name):")
    for field in record_set.fields:
        print(f"  - @id: {field.id}, name: {getattr(field, 'name', '<N/A>')}")


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# List all record set @ids
record_sets = [r.id for r in metadata.record_sets]
print("RecordSet @ids:", record_sets)

# Prepare dataframes for each record set
dataframes = {}
for record_set_id in record_sets:
    print(f"\nLoading data from RecordSet '@id': {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records. Columns:")
        print(df.columns.tolist())
        print("Sample data:")
        display(df.head())
    else:
        print("No records found or unable to load.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Set the target record set for EDA (choose the main data table)
target_record_set = record_sets[0] if record_sets else None

if target_record_set is not None and target_record_set in dataframes:
    df = dataframes[target_record_set]
    print(f"Working with RecordSet '@id': {target_record_set}")

    # Try to automatically select a numeric field
    numeric_field = None
    for col in df.columns:
        # try to guess by name
        if any(name in col.lower() for name in ['abundance', 'count', 'weight', 'coverage', 'norm', 'intensity', 'mw', 'pi']):
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field = col
                break
    # fallback: first numeric column
    if numeric_field is None:
        nums = df.select_dtypes(include=['number']).columns
        if len(nums):
            numeric_field = nums[0]
    if numeric_field is None:
        print("No numeric field found for EDA.")
    else:
        print(f"Using numeric field for EDA: {numeric_field}")

        # Filter: values > threshold
        threshold = df[numeric_field].mean() if pd.notnull(df[numeric_field].mean()) else 10
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} (z-score):")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a categorical column
        group_field = None
        for col in df.columns:
            if col != numeric_field and pd.api.types.is_object_dtype(df[col]):
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped mean {numeric_field} by {group_field}:")
            display(grouped_df.head())
        else:
            print("No categorical group field found.")
else:
    print("No valid RecordSet with data for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if target_record_set is not None and target_record_set in dataframes and numeric_field is not None:
    plt.figure(figsize=(8, 5))
    sns.histplot(dataframes[target_record_set][numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    # Scatter plot if grouping exists
    if 'group_field' in locals() and group_field is not None:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field, y=numeric_field, data=dataframes[target_record_set])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This notebook demonstrated how to load and explore the FAIR⁲ dataset on extracellular vesicle protein analysis using `mlcroissant`.

With Croissant, we:
- Loaded metadata and data records using record set and field `@id`s
- Identified record structures and available fields for analysis
- Performed basic filtering, normalization, and grouping on numeric fields
- Visualized data distributions

You can extend this notebook to perform more specific analyses or visualizations depending on your downstream research goals.

<!-- End of notebook -->